# Cathey — Evaluation Notebook

Measures the model's **intent-classification accuracy** on text inputs.

Each test case feeds a text string through the pipeline and checks whether
the output JSON `type` matches the expected label.

**Four intent types**

| Type | When |
|---|---|
| `direct_command` | User names a device + action ('turn on the light') |
| `needs_clarification` | User describes a feeling without specifying a device ('it's cold') |
| `general_qa` | Non-device question, greeting, or conversational filler |
| `invalid` | No 'Cathey' wake word detected (pre-filter catch) |

**Sections**
1. Bootstrap
2. Load Evaluation Data (`evaluation_data.json` — 4000 cases, 1000 per type)
3. Evaluation Runner
4. Evaluation Report
5. Custom Test Cases


## 1. Bootstrap

Loads the LLM, embedding model, memory, and agent.  
*(Skip if already initialized from `dev_debug.ipynb` in the same kernel.)*

In [ ]:
# Bootstrap — run once per kernel session
import sys, os, logging, time
logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
logging.getLogger('faster_whisper').setLevel(logging.ERROR)

# Add project root to path (needed when running from a subdirectory)
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from sentence_transformers import SentenceTransformer
from nlp.llm_parser import LLMParser
from core.memory import MemoryManager
from core.agent import CatheyAgent
from core.config import EMBED_MODEL_NAME

print('Loading LLM ...')
llm = LLMParser()

print('Loading embedding model ...')
embed = SentenceTransformer(EMBED_MODEL_NAME)
embed.encode('warmup', convert_to_numpy=True)

memory = MemoryManager(
    embed_fn=lambda t: embed.encode(t, convert_to_numpy=True).tolist()
)

# Stub GPIO for non-Pi environments
try:
    from hardware.gpio_executor import GPIOExecutor
    gpio = GPIOExecutor()
except Exception:
    gpio = None

# Stub TTS — prints instead of speaking (keeps evaluation output clean)
def _silent_speak(text):
    pass

agent = CatheyAgent(llm=llm, memory=memory, speak=_silent_speak, gpio=gpio)
print('\nAll components ready. Starting evaluation ...')

Loading LLM ...
Loading LLM (Qwen/Qwen2.5-3B-Instruct) on mps [torch.float16] ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading LoRA adapter from cathey_lora_adapter/final_adapter ...
LoRA adapter loaded.
LLM ready.
Loading embedding model ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


All components ready. Starting evaluation ...


## 2. Load Evaluation Data

Loads `evaluation_data.json` — **4000 cases, 1000 per intent type**.
Generated by a dedicated script covering diverse phrasings, wake-word variants,
and edge cases across all four categories.

You can also define extra cases in **Section 5** to append your own.


In [2]:
# ── Load evaluation_data.json ────────────────────────────────────────────────
import json as _json

EVAL_DATA_PATH = 'evaluation_data.json'

with open(EVAL_DATA_PATH, encoding='utf-8') as _f:
    _raw = _json.load(_f)

# Convert to list of (input, expected_type) tuples
EVAL_CASES = [(_d['input'], _d['expected_type']) for _d in _raw]

from collections import Counter
dist = Counter(exp for _, exp in EVAL_CASES)
print(f'Loaded {len(EVAL_CASES)} evaluation cases from {EVAL_DATA_PATH}')
print()
for t in ['direct_command', 'needs_clarification', 'general_qa', 'invalid']:
    print(f'  {t:<25} {dist[t]:>5} cases')


Loaded 4000 evaluation cases from evaluation_data.json

  direct_command             1000 cases
  needs_clarification        1000 cases
  general_qa                 1000 cases
  invalid                    1000 cases


## 3. Evaluation Runner

Runs all cases through the pipeline and collects results.
- No-wake-word inputs → caught by `contains_assistant_name()` prefilter → `invalid`
- Wake-word inputs → `llm.parse_unified()` for LLM classification
- No episodes saved during evaluation (clean, reproducible results)


In [ ]:
# ── Evaluation runner ────────────────────────────────────────────────────────
import time, pandas as pd
from core.agent import contains_assistant_name

# Merge with any custom cases from Section 5
try:
    all_cases = EVAL_CASES + USER_CASES
    print(f'Running {len(all_cases)} cases '
          f'({len(EVAL_CASES)} from file + {len(USER_CASES)} custom)')
except NameError:
    all_cases = EVAL_CASES
    print(f'Running {len(all_cases)} cases from evaluation_data.json')

print('This may take a while — sit tight...\n')

rows = []
n = len(all_cases)

for idx, (text, expected) in enumerate(all_cases):
    t0 = time.perf_counter()

    # Step 1: prefilter
    if not contains_assistant_name(text):
        got = 'invalid'
        ms  = (time.perf_counter() - t0) * 1000
    else:
        # Step 2: LLM classification (context='' for clean eval)
        result, _, _ = llm.parse_unified(text, context='', verbose=False)
        got = result.get('type', 'invalid')
        ms  = (time.perf_counter() - t0) * 1000

    match = got == expected
    rows.append({
        'input':      text,
        'expected':   expected,
        'got':        got,
        'match':      match,
        'latency_ms': round(ms, 1),
    })

    # Progress every 100 cases
    if (idx + 1) % 100 == 0:
        done = [r for r in rows]
        acc  = sum(r['match'] for r in done) / len(done)
        print(f'  [{idx+1:>4}/{n}]  running accuracy: {acc:.1%}')

df = pd.DataFrame(rows)
print(f'\nDone. {len(df)} cases evaluated.')


Running 4000 cases from evaluation_data.json
This may take a while — sit tight...

  [ 100/4000]  running accuracy: 97.0%
  [ 200/4000]  running accuracy: 95.0%
  [ 300/4000]  running accuracy: 94.3%
  [ 400/4000]  running accuracy: 93.8%
  [ 500/4000]  running accuracy: 93.6%
  [ 600/4000]  running accuracy: 93.8%
  [ 700/4000]  running accuracy: 94.4%
  [ 800/4000]  running accuracy: 94.6%
  [ 900/4000]  running accuracy: 94.0%
  [1000/4000]  running accuracy: 94.1%
  [1100/4000]  running accuracy: 93.9%
  [1200/4000]  running accuracy: 93.8%
  [1300/4000]  running accuracy: 94.1%
  [1400/4000]  running accuracy: 94.0%
  [1500/4000]  running accuracy: 93.9%
  [1600/4000]  running accuracy: 93.9%
  [1700/4000]  running accuracy: 93.9%
  [1800/4000]  running accuracy: 93.7%
  [1900/4000]  running accuracy: 93.6%
  [2000/4000]  running accuracy: 93.7%
  [2100/4000]  running accuracy: 93.7%
  [2200/4000]  running accuracy: 93.7%
  [2300/4000]  running accuracy: 93.7%
  [2400/4000]  runni

## 4. Evaluation Report

Generates a full accuracy + latency breakdown, confusion matrix, and failure analysis.


In [4]:
# ── Evaluation Report ────────────────────────────────────────────────────────
from collections import Counter
import pandas as pd

TYPES = ['direct_command', 'needs_clarification', 'general_qa', 'invalid']

# ── 1. Per-type accuracy & latency ───────────────────────────────────────────
print('=' * 72)
print('  CATHEY INTENT CLASSIFICATION — EVALUATION REPORT')
print('=' * 72)
print(f"  Model   : {llm.__class__.__name__} ({llm.tokenizer.name_or_path if hasattr(llm,'tokenizer') and llm.tokenizer else 'llama_cpp'})")
print(f"  Dataset : evaluation_data.json  ({len(df)} cases)")
print()

print('── Per-type Accuracy & Latency ─────────────────────────────────────────')
print(f"  {'Type':<22}  {'Accuracy':>8}  {'Correct':>7}  {'Total':>5}  {'Avg ms':>7}  Bar")
print('  ' + '-' * 65)

for t in TYPES:
    sub = df[df['expected'] == t]
    if len(sub) == 0:
        continue
    acc    = sub['match'].mean()
    avg_ms = sub['latency_ms'].mean()
    bar    = '█' * round(acc * 20) + '░' * (20 - round(acc * 20))
    print(f"  {t:<22}  {acc:>7.1%}  {sub['match'].sum():>7}  {len(sub):>5}  {avg_ms:>7.0f}  {bar}")

overall_acc = df['match'].mean()
overall_ms  = df['latency_ms'].mean()
print('  ' + '─' * 65)
print(f"  {'OVERALL':<22}  {overall_acc:>7.1%}  {df['match'].sum():>7}  {len(df):>5}  {overall_ms:>7.0f}")
print()

# ── 2. Confusion matrix ───────────────────────────────────────────────────────
print('── Confusion Matrix (row=expected, col=got) ─────────────────────────────')
short = {'direct_command':'direct', 'needs_clarification':'clarify',
         'general_qa':'qa', 'invalid':'invalid'}
cols  = [short[t] for t in TYPES]
header = f"  {'expected \\ got':<16}" + ''.join(f"  {c:>8}" for c in cols)
print(header)
print('  ' + '-' * (16 + 10 * len(TYPES)))
for t_exp in TYPES:
    sub = df[df['expected'] == t_exp]
    row = f"  {short[t_exp]:<16}"
    for t_got in TYPES:
        cnt = (sub['got'] == t_got).sum()
        row += f"  {cnt:>8}"
    print(row)
print()

# ── 3. Most common misclassifications ─────────────────────────────────────────
failures = df[~df['match']]
print('── Misclassification Summary ────────────────────────────────────────────')
if len(failures) == 0:
    print('  ✓ No failures — perfect score!')
else:
    print(f'  Total failures: {len(failures)} / {len(df)} ({len(failures)/len(df):.1%})')
    print()
    mc = Counter(zip(failures['expected'], failures['got'])).most_common(10)
    print(f"  {'expected → got':<45}  {'count':>5}")
    print('  ' + '-' * 52)
    for (exp, got), cnt in mc:
        print(f"  {exp:<22} → {got:<18}  {cnt:>5}")
    print()

    # Sample failures per type
    print('── Sample Failures (up to 3 per type) ──────────────────────────────────')
    for t in TYPES:
        sub_fail = failures[failures['expected'] == t].head(3)
        if len(sub_fail) == 0:
            continue
        print(f'  [{t}]')
        for _, row in sub_fail.iterrows():
            print(f"    ✗ got={row['got']:<22}  input: {row['input'][:60]}")
    print()

# ── 4. Latency distribution ───────────────────────────────────────────────────
print('── Latency Distribution (LLM calls only, prefilter excluded) ───────────')
llm_df = df[df['expected'] != 'invalid']   # invalid cases hit prefilter, near 0 ms
pct = llm_df['latency_ms']
print(f'  p50  = {pct.quantile(0.50):>7.0f} ms')
print(f'  p90  = {pct.quantile(0.90):>7.0f} ms')
print(f'  p99  = {pct.quantile(0.99):>7.0f} ms')
print(f'  max  = {pct.max():>7.0f} ms')
print()
print('=' * 72)
print(f'  Overall accuracy: {overall_acc:.1%}  |  Avg latency: {overall_ms:.0f} ms')
print('=' * 72)


  CATHEY INTENT CLASSIFICATION — EVALUATION REPORT
  Model   : LLMParser (Qwen/Qwen2.5-3B-Instruct)
  Dataset : evaluation_data.json  (4000 cases)

── Per-type Accuracy & Latency ─────────────────────────────────────────
  Type                    Accuracy  Correct  Total   Avg ms  Bar
  -----------------------------------------------------------------
  direct_command            96.9%      969   1000     2656  ███████████████████░
  needs_clarification       90.7%      907   1000     3020  ██████████████████░░
  general_qa                89.2%      892   1000     2332  ██████████████████░░
  invalid                  100.0%     1000   1000        0  ████████████████████
  ─────────────────────────────────────────────────────────────────
  OVERALL                   94.2%     3768   4000     2002

── Confusion Matrix (row=expected, col=got) ─────────────────────────────
  expected \ got      direct   clarify        qa   invalid
  --------------------------------------------------------
  

## 5. Custom Test Cases

Define `USER_CASES` below and re-run **Section 3** to include them in the results.


In [5]:
# ── Custom cases — edit and re-run Section 3 ─────────────────────────────────
USER_CASES = [
    # ('Cathey, close the curtain.',  'direct_command'),
    # ('Cathey, I feel a bit hot.',    'needs_clarification'),
    # ('Cathey, what year is it?',     'general_qa'),
    # ('Open the window.',             'invalid'),
]
print(f'{len(USER_CASES)} custom case(s) defined.')


0 custom case(s) defined.
